<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/8_01_Scripting_Spatial_Processing_via_GeoPandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8: Spatial processing with GeoPandas — Part 2

Today we build on Week 7 using **intersection and difference** to process building footprints and administrative boundaries for Kasungu, Malawi.

**Question:** How much mapped building-footprint area lies within each of our four study regions?

We will download two layers, inspect and project them, overlay polygons, compare results, and export a map and summary. Run examples in order. Task cells are for your own code.

If imports fail, run `%pip install geopandas matplotlib` in a separate cell and retry.


## GeoSeries recap

A `GeoSeries` is a one-dimensional array of geometries. Other attribute columns are ordinary Pandas Series. The points below are abstract Cartesian examples, so they have no geographic CRS.


In [ ]:
# Example: Creating a GeoPandas Series with Point geometries
import geopandas as gpd
from shapely.geometry import Point

# Creating a GeoPandas Series
points = gpd.GeoSeries([Point(0, 0), Point(1, 1), Point(2, 2)])
points

## GeoPandas GeoDataFrame

Remember that a `GeoPandas` `GeoDataFrame` is a two-dimensional data structure (e.g., a matrix) which extends the `Pandas` `DataFrame`, by including a `GeoPandas` `Series` as one of its columns, e.g., the geometry column.

Spatial information for each row in the `GeoDataFrame` is stored in this geometry column. See the example below to remind yourself.

In [ ]:
# Example: Creating a GeoPandas GeoDataFrame with Point geometries
import geopandas as gpd
from shapely.geometry import Point

# Here we specify a dictionary structure, and then convert to GeoDataFrame
# The dict key will be the column name
# The list of values the row information
data = {
    'Name': ['Location A', 'Location B', 'Location C'],
    'Type': ['Type D', 'Type F', 'Type X'],
    'geometry': [Point(2, 3), Point(4, 5), Point(0, 9)]
    }
gdf = gpd.GeoDataFrame(data)

# Inspect our GeoDataFrame
gdf

## Download and inspect two spatial layers

Download the original course ZIP from a fixed GitHub version. It contains four GADM administrative regions and an OpenStreetMap building sample for Kasungu. These are historical course samples, not a fresh query of current OSM data.

GeoPandas can read each shapefile directly inside the ZIP. We do not need manual uploads or extraction. The `zip://` paths below select individual layers.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import matplotlib.pyplot as plt

data_url = 'https://raw.githubusercontent.com/edwardoughton/spatial_computing/510dcb3b7d6954999434df162b077dabe2a98891/data/kasungu.zip'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
data_path = data_dir / 'kasungu.zip'
urlretrieve(data_url, data_path)
print(f'Downloaded {data_path.stat().st_size:,} bytes from {data_url}')


The boundary sample contains four GADM level 3 regions. Read its stored CRS and inspect the identifiers. See [GADM](https://gadm.org/) for the boundary data source; retain attribution and check its terms before reusing data outside class.


In [ ]:
boundaries_source = gpd.read_file(f'zip://{data_path.resolve().as_posix()}!kasungu_gadm3_regions.shp')
if boundaries_source.crs is None:
    raise ValueError('Boundary CRS is missing.')
print('Boundary CRS:', boundaries_source.crs)
boundaries_source.head()

# Use one local projected CRS in meters for both layers.
analysis_crs = boundaries_source.estimate_utm_crs()
boundaries = boundaries_source.to_crs(analysis_crs)
print('Analysis CRS:', analysis_crs)


We can utilize the `.plot()` function to visualize these geometries.

The sample below shows this graphically.

In [ ]:
# Example: Plotting our boundary geometries
boundaries.plot(alpha=0.5, edgecolor='k')

This boundary data suggests we have 4 regions within Kasungu Boma.

We can access the geographic identifier string IDs as follows.

In [ ]:
# Example: Printing the GID_3 column
print(boundaries['GID_3'])

Read the building-footprint sample, originally obtained from OpenStreetMap through [Geofabrik](https://download.geofabrik.de/). Reproject it to match the boundaries before overlaying. Missing buildings in OSM do not mean no buildings exist on the ground.


In [ ]:
buildings_source = gpd.read_file(f'zip://{data_path.resolve().as_posix()}!kasungu_buildings.shp')
if buildings_source.crs is None:
    raise ValueError('Building CRS is missing.')
# Keep footprint attributes; obtain region IDs from the boundary overlay.
# The sample already contains administrative fields, which would collide.
buildings = buildings_source[['osm_id', 'fclass', 'geometry']].copy().to_crs(analysis_crs)
# A stable ID for each original sample record lets us track clipped pieces.
buildings['building_id'] = range(len(buildings))
for name, layer in [('Boundaries', boundaries), ('Buildings', buildings)]:
    print(name, 'records:', len(layer), 'CRS:', layer.crs)
    print(layer.geom_type.value_counts())
    print('Missing:', layer.geometry.isna().sum(), 'Empty:', layer.geometry.is_empty.sum(), 'Invalid:', (~layer.geometry.is_valid).sum())
    if layer.geometry.isna().any() or layer.geometry.is_empty.any() or not layer.geometry.is_valid.all():
        raise ValueError(f'{name} has geometry issues; inspect before overlaying.')
buildings.head(3)


Plot the building geometries and print the actual record count. These are mapped footprints; completeness is unknown. Overlay both layers on a shared axis to check that their locations agree.


In [ ]:
print('Original building records:', len(buildings))
ax = boundaries.plot(facecolor='none', edgecolor='red', figsize=(10, 8))
buildings.plot(ax=ax, color='gray')
ax.set_title('Kasungu study boundaries and mapped building footprints')
ax.set_axis_off()
plt.show()


## Intersection

Intersection retains the overlapping portions of polygons and attributes from both layers. A building crossing a region boundary may be clipped or split. Output rows therefore should not automatically be called a count of whole buildings.

Start with region `MWI.9.1.6_1`.


In [ ]:
boundary = boundaries.loc[boundaries['GID_3'] == 'MWI.9.1.6_1'].copy()
if len(boundary) != 1:
    raise ValueError('Expected one selected boundary; inspect GID_3.')
ax = boundaries.plot(facecolor='none', edgecolor='gray')
boundary.plot(ax=ax, alpha=0.5, edgecolor='black')
plt.show()


Next, we can carry out the actual intersection.

Here, we are following the `GeoPandas` documentation page on set operations: https://geopandas.org/en/stable/docs/user_guide/set_operations.html

In this case, we will utilize the `.overlay()` function and specify "intersection".

Inspect and understand the difference below once this intersection has been carried out.  


In [ ]:
buildings_subset = buildings.overlay(boundary, how='intersection', keep_geom_type=True)
buildings_subset['footprint_m2'] = buildings_subset.geometry.area
print('Output rows:', len(buildings_subset))
print('Unique source building records:', buildings_subset['building_id'].nunique())
print('Clipped footprint area (m²):', buildings_subset['footprint_m2'].sum())
ax = boundary.plot(facecolor='none', edgecolor='red', figsize=(10, 8))
buildings_subset.plot(ax=ax, color='gray')
ax.set_title('Building footprints clipped to the selected region')
plt.show()


## Task: intersect two regions

Select two of the four region IDs, then intersect the buildings with that selection.

* Report original records, output rows, and unique source `building_id` values.
* Calculate clipped footprint area in square meters.
* Map the result and export it to GeoPackage.
* Explain why one source building could appear in both regions and why output row count is not always a whole-building count.


In [ ]:
# Enter your attempt here


## Difference

Difference retains the portions of the left layer outside the right layer. A partially overlapping building is clipped, rather than removed in its entirety. Changing layer order changes the result.

Use the same selected boundary for intersection and difference, so we can check that their areas account for the original footprint area.


In [ ]:
buildings_outside = buildings.overlay(boundary, how='difference', keep_geom_type=True)
print('Difference output rows:', len(buildings_outside))
original_area = buildings.geometry.area.sum()
inside_area = buildings_subset.geometry.area.sum()
outside_area = buildings_outside.geometry.area.sum()
print('Original footprint area (m²):', original_area)
print('Inside + outside (m²):', inside_area + outside_area)
print('Area residual (m²):', original_area - inside_area - outside_area)
ax = buildings_outside.plot(color='gray', figsize=(10, 8))
boundary.plot(ax=ax, facecolor='none', edgecolor='red')
ax.set_title('Building footprint portions outside the selected region')
plt.show()


## Task: compare all four regions

Intersect the original buildings with all four boundaries. Calculate clipped `footprint_m2`, then group by `GID_3` to report unique source building records and total footprint area.

* Retain any region with no mapped buildings in your final summary.
* Join region area in km² and calculate mapped footprint m² per km².
* Check whether boundaries overlap before interpreting summed results.
* Export a summary CSV and overlay GeoPackage; map the regional pattern.
* Write a numerical finding and a limitation. These metrics describe the mapped sample; they are not population counts or proof of OSM completeness.

**Extension: acquire a new dataset.** Choose a small area from a [Geofabrik download](https://download.geofabrik.de/) and obtain a compatible boundary. Record the exact download URL and date, inspect archive filenames, read the selected layers, align CRS, and clip roads or points of interest to your study area. Start with a small extract to keep processing manageable. Inspect actual boundary names before defining a capital region; administrative units may extend beyond the city.


In [ ]:
# Enter your attempt here


## Task: building floorspace scenarios

Use the clipped footprints for your selected region to calculate mapped ground-floor area in m². Estimate total floorspace under uniform assumptions of 2, 3, and 4 floors per building.

Export a scenario table. Explain that these are assumed floor counts, not observed heights, and that footprints crossing the boundary contribute only their clipped portions. What additional data would improve your estimate?


In [ ]:
# Enter your attempt here


## Before you finish

Submit a regional summary CSV, a map, and a processed GeoPackage. Include the source URL, analysis CRS, and a short interpretation of a finding, an assumption, and a limitation.

Intersection creates overlap geometry; difference creates outside geometry. A spatial join instead attaches attributes based on location without clipping the left geometries. See [GeoPandas overlay documentation](https://geopandas.org/en/stable/docs/user_guide/set_operations.html).
